# ir_calendar_init_tables — 一次性建表（bronze 2 張 + silver 4 張）

- 用途：為法說行事曆（ir_calendar）批次消費 job 建立 Delta table。爬蟲每批產出除了落 Volume、轉送系統 API，也在 Databricks 留一份：bronze 原樣保存、silver 供查詢。
- 執行次數：一次。可安全重跑：全部 `CREATE TABLE IF NOT EXISTS`，不動既有資料；只有 `recreate = YES_DROP_ALL` 才會 DROP。
- 輸入 table：無。
- 輸出 table：`{catalog}.{schema}.b_{domain}_*` 2 張、`s_{domain}_*` 4 張，清單見 [c02]。`domain` 預設 `ir_calendar`。
- 參數（widgets）：`catalog`、`schema`、`domain`、`recreate`。預設值對照 `config/project.yml`。
- 排程：無（手動執行一次）。
- 負責人 / 更新日期：（填）/ 2026-09-21
- 設計說明：`docs/20260921_ir_calendar_lakehouse_design.md`；分層通則：`docs/conventions.md` 2.1。

## 分層

| 層 | 表 | 性質 | 內容 |
|---|---|---|---|
| bronze | `b_ir_calendar_record` | **append-only**，schema 固定 | 爬蟲批次裡每個檔案一列：`record_type` + `payload`（JSON 全文，STRING）+ 批次血緣。所有 record_type 共用這一張 |
| bronze | `b_ir_calendar_batch_log` | 以 `batch_id` MERGE | 每批的 manifest 摘要、處理結果、API 回應（job 遙測，不是來源資料，所以允許覆蓋） |
| silver | `s_ir_calendar_conference` | MERGE，鍵 `(company_key, period, revision)` | 法說場次，每個 revision 一列 |
| silver | `s_ir_calendar_summary` | MERGE，鍵 `(company_key, period)` | 法說彙整（系統端無對應表） |
| silver | `s_ir_calendar_document_file` | MERGE，鍵 `volume_path` | 實體檔案索引（Volume 上每個 PDF / HTM） |
| silver | `s_ir_calendar_company` | MERGE，鍵 `company_key` | 公司主表快照 |

silver 全部可以從 bronze 重算（消費 job 的 `rebuild_silver = true`），不必碰 Volume 歸檔。

## 維護者須知

| 事項 | 說明 |
|---|---|
| 表命名 | `<層級>_<領域>_<短名>`：`b_` 原始、`s_` 銀質、`g_` 金質、`app_` 其他系統來的。這裡的層級前綴固定，只有 `domain` 可改。 |
| 爬蟲加欄位 | bronze **不用改**（`payload` 整包收）。silver 想用新欄位才改：`ALTER TABLE ADD COLUMNS` → 消費 job [c04] payload schema → [c06] transform。不要 DROP 重建。 |
| 爬蟲加 record_type | bronze 自動收（未登記的 record_type 一樣進 `b_*_record`）。要查才建 silver 表：加 DDL cell、[c04] `TABLE_KEYS`、[c06] transform 並登記到 `SILVER_TRANSFORMS`。 |
| 主鍵 | Delta 沒有 PK 約束；silver 的鍵寫在各 DDL cell 註解，消費 job [c04] `TABLE_KEYS` 用同一組鍵 MERGE。**改鍵要兩邊一起改。** |
| 不分割區 | 都是小表（bronze 每天數十列）。bronze 用 liquid clustering `CLUSTER BY (record_type)`；silver 開 autoOptimize。 |
| 時間欄位 | `TIMESTAMP` 一律存 UTC 瞬間（爬蟲的 `+08:00` 與無時區字串在 silver 轉換時統一處理）。`DATE`（含 `conference_date`）為**台北曆日**：爬蟲端已統一換算成台北時區，Databricks 不做任何時區處理。 |
| 直接查 bronze | Databricks SQL 可用路徑運算子：`SELECT payload:conference_date, payload:company_name FROM b_ir_calendar_record WHERE record_type = 'ir_conference'`。 |
| 對應文件 | `ir_calendar/doc/20260918_pass_data_to_databricks.md` 第 7 節（manifest 格式）、第 9 節（各類產出）；`doc/20260825_系統後端DB的Schema規劃.md`（系統端表，本 notebook 不建那些表）。 |


In [ ]:
# [c01] params
# 預設值請對照 config/project.yml；job 執行時由 job parameters 覆蓋。
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "")
dbutils.widgets.text("domain", "ir_calendar")      # 表名中段；層級前綴 b_ / s_ 固定，見 docs/conventions.md 2.1
# 危險開關：只有填 YES_DROP_ALL 才會先 DROP 再建（資料全丟）。日常請留空。
dbutils.widgets.text("recreate", "")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
domain = dbutils.widgets.get("domain").strip()
recreate = dbutils.widgets.get("recreate").strip() == "YES_DROP_ALL"
assert catalog and schema and domain, "catalog / schema / domain 不可為空"
print(f"目標 schema = {catalog}.{schema}，domain = {domain!r}，recreate = {recreate}")


In [ ]:
# [c02] imports
# 表清單（順序即建立順序）：(層級, 短名)。完整表名 = <層級前綴><domain>_<短名>，如 b_ir_calendar_record、s_ir_calendar_conference。
LAYER_PREFIX = {"bronze": "b_", "silver": "s_"}
TABLES = [
    ("bronze", "record"),          # 爬蟲批次每個檔案一列，payload = JSON 全文；append-only
    ("bronze", "batch_log"),       # 每批處理紀錄（manifest 摘要 + 處理結果 + API 回應）
    ("silver", "conference"),      # calendar/*.json：法說場次，每個 revision 一列
    ("silver", "summary"),         # summaries/*.json：法說彙整
    ("silver", "document_file"),   # 實體檔案索引：落到 Volume 的每個 PDF / HTM
    ("silver", "company"),         # master/company.json：公司主表快照
]
DDLS: dict[tuple[str, str], str] = {}      # (層級, 短名) → CREATE TABLE 語句，由 [c10]～[c15] 填入


def full_name(layer: str, short: str) -> str:
    return f"{catalog}.{schema}.{LAYER_PREFIX[layer]}{domain}_{short}"


# 小表共用屬性：自動合併小檔案；不分割區。
TBLPROPS = (
    "TBLPROPERTIES ("
    "'delta.autoOptimize.optimizeWrite' = 'true', "
    "'delta.autoOptimize.autoCompact' = 'true')"
)


In [ ]:
# [c10] ddl_b_record
# bronze 主表：爬蟲批次 manifest.files[] 的每個檔案一列（外加 manifest 本身一列，record_type = batch_manifest）。
# append-only；重跑同一批時先 DELETE WHERE batch_id 再 append（冪等）。schema 不隨爬蟲欄位變動：新欄位都在 payload 裡。
# 沒有主鍵；(batch_id, batch_path) 在正常情況下唯一。
DDLS[("bronze", "record")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("bronze", "record")} (
  batch_id     STRING    NOT NULL COMMENT '批次目錄名 <6碼序號>_<yyyyMMddTHHmmss>',
  seq          INT       NOT NULL COMMENT '批次序號（manifest.seq）；silver 取最新版本靠它排序',
  record_type  STRING    NOT NULL COMMENT 'batch_manifest / ir_conference / ir_summary / ir_document / ir_document_file / api_payload / app_company / app_company_category …',
  batch_path   STRING    NOT NULL COMMENT '批次內相對路徑，如 calendar/2026-09-18_KRX005930_2026Q3.json',
  company_key  STRING             COMMENT '<市場>:<代號>（manifest 條目有帶才填）',
  period       STRING             COMMENT '曆年期別 yyyyQn（manifest 條目有帶才填）',
  sha256       STRING             COMMENT '檔案內容雜湊（manifest.files[].sha256，下載時已核對）',
  bytes        BIGINT             COMMENT '檔案大小',
  volume_path  STRING             COMMENT '實體檔案（ir_document_file）落地的 Volume 完整路徑；JSON 類為 NULL',
  payload      STRING    NOT NULL COMMENT 'JSON 全文：JSON 檔是檔案內容；實體檔是 manifest 該條目。可用 payload:欄位 路徑查詢',
  ingested_at  TIMESTAMP NOT NULL COMMENT '寫入時間（UTC）'
)
USING DELTA
CLUSTER BY (record_type)
COMMENT 'ir_calendar bronze：爬蟲批次原樣落地，一檔一列，append-only'
"""


In [ ]:
# [c11] ddl_b_batch_log
# 主鍵：batch_id。job 遙測（不是來源資料），所以允許 MERGE 覆蓋：失敗先寫 FAILED，重跑成功後覆蓋成 SUCCESS。
# API 回應併在 api_results 陣列，不另開表；查失敗明細用 explode(api_results)。
DDLS[("bronze", "batch_log")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("bronze", "batch_log")} (
  batch_id         STRING    NOT NULL COMMENT '批次目錄名，主鍵',
  seq              INT       NOT NULL COMMENT '批次序號（manifest.seq）',
  prev_seq         INT                COMMENT '前一批序號；第一批為 NULL',
  kind             STRING             COMMENT 'incremental（日常）/ master（主檔全量快照）',
  producer_mode    STRING             COMMENT '爬蟲執行模式：daily_scan / post_event / fetch_docs / master',
  producer_host    STRING             COMMENT '爬蟲主機名',
  producer_git_sha STRING             COMMENT '爬蟲程式版本',
  generated_at     TIMESTAMP          COMMENT '爬蟲端產出時間（UTC）',
  counts           MAP<STRING, INT>   COMMENT '各 record_type 筆數（manifest.counts）',
  files_total      INT                COMMENT 'manifest.files 總數',
  landed_files     INT                COMMENT '落到 Volume 的實體檔案數',
  archived_files   INT                COMMENT '歸檔到 <volume_root>/ir_calendar/batches/ 的 JSON 數',
  bronze_rows      INT                COMMENT '寫進 b_*_record 的列數',
  api_results      ARRAY<STRUCT<
                     endpoint: STRING, idempotency_key: STRING, rows: INT, http_status: INT, success: BOOLEAN,
                     created: INT, updated: INT, unchanged: INT, failed: INT,
                     failures_json: STRING, response_json: STRING, posted_at: TIMESTAMP>>
                                      COMMENT '每次 POST 系統 API 的結果；未轉送為空陣列',
  status           STRING    NOT NULL COMMENT 'SUCCESS / FAILED',
  error            STRING             COMMENT '失敗時的例外訊息（前 2000 字）',
  job_run_id       STRING             COMMENT 'Databricks job run id（由 job parameter 帶入）',
  processed_at     TIMESTAMP NOT NULL COMMENT '本 job 處理時間（UTC）'
)
USING DELTA
COMMENT 'ir_calendar bronze：每個爬蟲批次一列的處理紀錄'
{TBLPROPS}
"""


In [ ]:
# [c12] ddl_s_conference
# 主鍵：(company_key, period, revision)。同一場次改期或補連結會出新 revision，保留歷史；
# 要「目前版本」取每個 (company_key, period) 的 max(revision)。系統端的鍵是 (stock_code, fiscal_period = period)。
# 來源：bronze record_type = ir_conference，由消費 job [c06] transform_conference 產生。
DDLS[("silver", "conference")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("silver", "conference")} (
  company_key          STRING    NOT NULL COMMENT '<市場>:<代號>，主鍵之一',
  company_name         STRING             COMMENT '公司名稱',
  english_name         STRING             COMMENT '英文名',
  stock_code           STRING             COMMENT '股票代號',
  market               STRING             COMMENT '交易所代碼',
  category             STRING             COMMENT '公司分類代碼',
  period               STRING    NOT NULL COMMENT '曆年期別 yyyyQn（如 2026Q2），主鍵之一；即系統端 fiscal_period',
  fiscal_period        STRING             COMMENT '爬蟲舊欄位：可能只有 Qn 不帶年，勿當鍵',
  source_fiscal_period STRING             COMMENT '來源公司自稱的期別（如 FY2026Q3），可為 NULL',
  conference_date      DATE               COMMENT '法說日期：台北曆日（爬蟲端已統一換算，本 job 不做任何時區處理）',
  start_time           STRING             COMMENT '開始時間（多為 NULL）',
  end_time             STRING             COMMENT '結束時間（實測 0% 可得）',
  conference_type      STRING             COMMENT 'ONSITE / ONLINE / PHONE / HYBRID / NULL',
  location             STRING             COMMENT '實體地點',
  meeting_link         STRING             COMMENT '線上會議連結',
  document_url         STRING             COMMENT '簡報 / 文件連結',
  status               STRING             COMMENT '爬蟲一律 SCHEDULED；COMPLETED 由系統端翻轉',
  importance           INT                COMMENT '重要性 1~5，由分類推導的初始值',
  source               STRING             COMMENT '來源：MOPS / TWSE / TPEX / LLM_SEARCH / SEED_ONETIME …',
  source_url           STRING             COMMENT '來源網址',
  confidence           STRING             COMMENT 'CONFIRMED（官方）/ TENTATIVE（轉述）',
  remark               STRING             COMMENT '備註',
  recipients           ARRAY<STRING>      COMMENT '通知對象代碼',
  revision             INT       NOT NULL COMMENT '事件版本號，主鍵之一',
  crawled_at           TIMESTAMP          COMMENT '爬取時間（UTC）',
  batch_id             STRING             COMMENT '來源批次（bronze 血緣）',
  seq                  INT                COMMENT '來源批次序號',
  batch_path           STRING             COMMENT '批次內路徑',
  updated_at           TIMESTAMP NOT NULL COMMENT 'silver 寫入 / 更新時間（UTC）'
)
USING DELTA
COMMENT 'ir_calendar silver：法說場次（對應系統端 app_ir_conference；每個 revision 一列）'
{TBLPROPS}
"""


In [ ]:
# [c13] ddl_s_summary
# 主鍵：(company_key, period)，同鍵以較新的批次為準。系統端沒有對應表，這裡是唯一的結構化存放處（規劃書第 9 節「不要丟」）。
# summary 物件欄位由 LLM 產出、可能增減：展開常用欄位，另存 summary_json 原文（也可回 bronze 取 payload）。
DDLS[("silver", "summary")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("silver", "summary")} (
  company_key     STRING    NOT NULL COMMENT '<市場>:<代號>，主鍵之一',
  company_name    STRING             COMMENT '公司名稱',
  stock_code      STRING             COMMENT '股票代號',
  market          STRING             COMMENT '交易所代碼',
  category        STRING             COMMENT '公司分類代碼',
  period          STRING    NOT NULL COMMENT '曆年期別 yyyyQn，主鍵之一',
  fiscal_period   STRING             COMMENT '爬蟲舊欄位，勿當鍵',
  conference_date DATE               COMMENT '法說日期：台北曆日（爬蟲端已統一換算，本 job 不做任何時區處理）',
  fallback        BOOLEAN            COMMENT 'true = 找不到法說內容、改以財報新聞稿彙整',
  found           BOOLEAN            COMMENT 'summary.found',
  core_points     ARRAY<STRING>      COMMENT '重點',
  guidance        STRING             COMMENT '展望 / 指引',
  key_numbers     ARRAY<STRING>      COMMENT '關鍵數字',
  risks           ARRAY<STRING>      COMMENT '風險',
  notes           STRING             COMMENT '彙整備註（期別對應、來源限制等）',
  sources         ARRAY<STRUCT<title: STRING, url: STRING, media: STRING>> COMMENT '引用來源',
  summary_json    STRING             COMMENT 'summary 物件原文（JSON 字串）',
  recipients      ARRAY<STRING>      COMMENT '通知對象代碼',
  crawled_at      TIMESTAMP          COMMENT '彙整時間（UTC）',
  batch_id        STRING             COMMENT '來源批次（bronze 血緣）',
  seq             INT                COMMENT '來源批次序號',
  batch_path      STRING             COMMENT '批次內路徑',
  updated_at      TIMESTAMP NOT NULL COMMENT 'silver 寫入 / 更新時間（UTC）'
)
USING DELTA
COMMENT 'ir_calendar silver：法說彙整（LLM 產出；系統端無對應表）'
{TBLPROPS}
"""


In [ ]:
# [c14] ddl_s_document_file
# 主鍵：volume_path（Volume 上的完整路徑）。同名檔案 = 新 revision 覆蓋，這裡也覆蓋成最新的 sha256；舊版仍在 bronze。
# 公司 / 分類 / 期別 / 種類都取自 manifest 欄位，不拆檔名（規劃書 16.4）。
DDLS[("silver", "document_file")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("silver", "document_file")} (
  volume_path  STRING    NOT NULL COMMENT 'Volume 完整路徑 <volume_root>/<分類目錄>/<代稱>/<檔名>，主鍵',
  file_name    STRING             COMMENT '交付檔名 <代稱>_Q<季>CY<年>_<種類>.<副檔名>',
  company_key  STRING             COMMENT '<市場>:<代號>',
  company_slug STRING             COMMENT '公司代稱（Volume 目錄名）',
  category     STRING             COMMENT '公司分類代碼：PANEL_PEER / CUSTOMER / SUPPLIER',
  period       STRING             COMMENT '曆年期別 yyyyQn',
  doc_kind     STRING             COMMENT 'Presentation / Financial_Statements / Press_Release / Earnings_Transcript',
  fiscal_label STRING             COMMENT '公司口徑期別（如 FY2026Q3），可為 NULL',
  sha256       STRING             COMMENT '檔案內容雜湊（下載時已核對）',
  bytes        BIGINT             COMMENT '檔案大小',
  source_file  STRING             COMMENT '爬蟲工作面原檔名',
  source_url   STRING             COMMENT '原始下載網址',
  doc_date     DATE               COMMENT '文件日期，可為 NULL',
  batch_id     STRING             COMMENT '來源批次（bronze 血緣）',
  seq          INT                COMMENT '來源批次序號',
  batch_path   STRING             COMMENT '批次內路徑，如 docs/HK_1070/2026Q2/TCL-Electronics_Q2CY26_Presentation.pdf',
  updated_at   TIMESTAMP NOT NULL COMMENT 'silver 寫入 / 更新時間（UTC）'
)
USING DELTA
COMMENT 'ir_calendar silver：實體檔案索引，Volume 上每個 PDF / HTM 一列'
{TBLPROPS}
"""


In [ ]:
# [c15] ddl_s_company
# 主鍵：company_key（<市場>:<代號>，如 TPE:2353）。系統端用 stock_code 當鍵，但爬蟲的不變鍵是 company_key。
# 來源：bronze record_type = app_company（主檔批次的 master/company.json，rows[] 展開）；被移出名單的公司不會自動刪除，看 is_active。
# 公司分類常數（app_company_category）不另建 silver：只有 3 列，category_name 已在本表；要看原文查 bronze。
DDLS[("silver", "company")] = f"""
CREATE TABLE IF NOT EXISTS {full_name("silver", "company")} (
  company_key         STRING    NOT NULL COMMENT '<市場>:<股票代號>，如 TPE:2353、NYSE:HPQ，主鍵',
  company_name        STRING             COMMENT '公司名稱（台股為中文法定名）',
  english_name        STRING             COMMENT '英文名（TWSE 來源可能被截為 40 字）',
  stock_code          STRING             COMMENT '股票代號；系統 API 的 upsert 鍵',
  market              STRING             COMMENT '交易所代碼：TPE / TYO / SZ / SH / KRX / HK / NYSE / NASDAQ / ETR',
  market_type         STRING             COMMENT 'LISTED / OTC / EMERGING；海外多為硬套或 NULL',
  category_name       STRING             COMMENT '公司分類代碼：PANEL_PEER / CUSTOMER / SUPPLIER',
  industry            STRING             COMMENT '產業別（台股官方代碼、海外人工分類，基準不一）',
  website_url         STRING             COMMENT '官網',
  ir_url              STRING             COMMENT '投資人關係頁',
  aliases             ARRAY<STRING>      COMMENT '別名清單（不唯一，不可當鍵）',
  recipients          ARRAY<STRING>      COMMENT '通知對象代碼（A00 / B00 …）',
  remark              STRING             COMMENT '備註',
  is_active           BOOLEAN            COMMENT '是否仍追蹤；已下市為 false',
  profile_source      STRING             COMMENT '主檔資料來源說明',
  file_slug           STRING             COMMENT '交付檔名用的公司代稱（data/company_slugs.json），可為 NULL',
  master_generated_at TIMESTAMP          COMMENT '主檔產出時間（UTC）',
  batch_id            STRING             COMMENT '來源批次（bronze 血緣）',
  seq                 INT                COMMENT '來源批次序號',
  batch_path          STRING             COMMENT '批次內路徑（master/company.json）',
  updated_at          TIMESTAMP NOT NULL COMMENT 'silver 寫入 / 更新時間（UTC）'
)
USING DELTA
COMMENT 'ir_calendar silver：公司主表快照（對應系統端 app_company；以 company_key 為鍵）'
{TBLPROPS}
"""


In [ ]:
# [c20] run_create
# 建表順序照 TABLES。recreate 只在明確填 YES_DROP_ALL 時才 DROP，且先印出要刪的表。
missing = [t for t in TABLES if t not in DDLS]
assert not missing, f"缺 DDL：{missing}（對應 cell 沒跑到）"

if recreate:
    print("!!! recreate = YES_DROP_ALL，將刪除以下表與其資料：")
    for layer, short in TABLES:
        print("   ", full_name(layer, short))
    for layer, short in TABLES:
        spark.sql(f"DROP TABLE IF EXISTS {full_name(layer, short)}")

for layer, short in TABLES:
    spark.sql(DDLS[(layer, short)])
    print(f"ok  {full_name(layer, short)}")


In [ ]:
# [c30] check_tables
# 驗證：每張表都存在、欄位數、目前列數（都是小表，count 可接受）。
rows = []
for layer, short in TABLES:
    t = full_name(layer, short)
    df_t = spark.table(t)
    rows.append((t, len(df_t.columns), df_t.count()))
w = max(len(r[0]) for r in rows)
for t, ncol, nrow in rows:
    print(f"{t:<{w}}  cols={ncol:>2}  rows={nrow}")
